# Fitroom try-on spike on a free Colab GPU

This notebook runs Fitroom's try-on spike (`script/modal/tryon_engine.py`): two open-source models, FASHN VTON 1.5 and Leffa, paint a shop's photo of a garment onto Fitroom's mannequin render, and a three-gate validator says whether the page could show each result. You get a contact sheet (`sheet.png`) and the scores (`results.json`), the same files the Modal version makes.

- Google gives a **T4 GPU for free** here. No account or card beyond a Google login.
- **Nothing of the person is used.** The inputs are the mannequin render and the shop's own photo. No photo of anyone, no measurements.
- **Time:** about 10 minutes to install and download the weights (around 10 GB, again every new session), then 30 to 90 seconds per picture per model on the T4.

Runtime → Run all. It stops once to ask for your files: pick the `spike.zip` that `node script/spike/inputs.mjs` made on the Mac (docs/ai-spike.md, "Making the inputs"). Running it all again is safe: the upload cell starts from an empty folder and the run cell pulls the latest engine from GitHub first.

## 1. Check the GPU

If this prints a table with **Tesla T4**, carry on. If it says no GPU: Runtime → Change runtime type → **T4 GPU** → Save, then run again.

In [ ]:
!nvidia-smi || echo "No GPU. Runtime > Change runtime type > T4 GPU > Save, then run this cell again." 

## 2. Install

Three cells: Fitroom's code, the two models with their libraries, the weights. About 10 minutes in all. Colab's own `torch` is kept; nothing is reinstalled over it.

In [ ]:
# Fitroom's engine. The repo is public; if the clone fails (made private?), zip the folder
# script/modal from your copy and upload it when asked.
import io, os, shutil, subprocess, zipfile

ENGINE = "/content/fitroom/script/modal"
if not os.path.isdir(ENGINE):
    ok = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ac7552/fitroom", "/content/fitroom"]).returncode == 0
    if not ok:
        print("Could not clone. Upload a zip of script/modal (tryon_engine.py and the files next to it).")
        from google.colab import files
        for data in files.upload().values():
            zipfile.ZipFile(io.BytesIO(data)).extractall("/content/upload")
        found = next(root for root, _, names in os.walk("/content/upload") if "tryon_engine.py" in names)
        shutil.copytree(found, ENGINE)
print("engine at", ENGINE, "->", sorted(n for n in os.listdir(ENGINE) if n.endswith(".py")))

In [ ]:
%%bash
set -e
cd /content
# The two models, at the commits the engine was read against.
[ -d fashn-vton-1.5 ] || git clone -q https://github.com/fashn-AI/fashn-vton-1.5
(cd fashn-vton-1.5 && git checkout -q 7c0f10af3f91ad4048fe9729c470a13ef905d25a)
[ -d Leffa ] || git clone -q https://github.com/franciszzj/Leffa
(cd Leffa && git checkout -q 05a259104b6927607776c7edb3e86b75406f20aa)
# Libraries, pinned as the Modal image pins them (script/modal/tryon_spike.py). torch, torchvision, numpy,
# opencv, pillow, scipy and scikit-image stay as Colab ships them. onnxruntime-gpu runs DWPose on the GPU
# (the CPU build, if present, would shadow it).
pip uninstall -y -q onnxruntime 2>/dev/null || true
pip install -q diffusers==0.31.0 transformers==4.46.3 accelerate==1.1.1 safetensors==0.4.5 "huggingface_hub>=0.26.2,<1.0" \
  einops==0.8.0 timm==1.0.11 kornia==0.7.4 "onnxruntime-gpu>=1.20" \
  fvcore==0.1.5.post20221221 iopath==0.1.10 omegaconf==2.3.0 yacs==0.1.8 pycocotools==2.0.8 \
  cloudpickle==3.1.0 tabulate==0.9.0 termcolor==2.5.0 av
# --no-deps: their dependency lists would pull another torch or opencv.
pip install -q --no-deps fashn-human-parser==0.1.1
pip install -q --no-deps ./fashn-vton-1.5
# Leffa's vendored detectron2 runs the R50-FPN DensePose model on torchvision's ops; no compiled _C needed.
python -c "import torch, fashn_vton, onnxruntime; print('torch', torch.__version__, '| onnxruntime providers', onnxruntime.get_available_providers())"


In [ ]:
# The weights, into /content/weights: FASHN (the same files as its scripts/download_weights.py), the parts
# of franciszzj/Leffa the try-on reads, and the validator's BiRefNet and DINOv2. About 10 GB.
import os, sys
os.environ["HF_HOME"] = "/content/weights/hf"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ.setdefault("FASHN_PARSER", "off")   # the research-only parser isn't fetched; step 4 can turn it on
sys.path[:0] = [p for p in ("/content/fitroom/script/modal", "/content/Leffa") if p not in sys.path]
import tryon_engine as engine
engine.download("/content/weights")
print("done:", sorted(os.listdir("/content/weights")))

## 3. Your pictures

Upload the sheet's inputs: the one `spike.zip` from `node script/spike/inputs.mjs ...` on the Mac, or the loose files (docs/ai-spike.md, "The sheet folder"). For each NAME:

- `NAME-person.png` — the mannequin render (needed)
- `NAME-garment.jpg` (or .png / .webp) — the shop's photo (needed)
- `NAME-category.txt` — `tops` or `bottoms` (else tops)
- `NAME-outline.png`, `NAME-mask.png`, `NAME-print.png`, `NAME-pose.json` — optional, see the doc

Pick them all in one go in the file dialog (a zip and loose files together is fine too). The folder is emptied first, so re-running this cell replaces the last upload rather than adding to it, and a browser's ` (1)` copy suffix is taken off the names. If the files are in Google Drive instead, run `from google.colab import drive; drive.mount('/content/drive')` and point `SPIKE` at that folder.

In [ ]:
import io, os, re, shutil, zipfile
from google.colab import files

SPIKE = "/content/spike"
shutil.rmtree(SPIKE, ignore_errors=True)   # a fresh folder each time: no leftovers from the last upload
os.makedirs(SPIKE)

def plain(name):
    """"tee-person (1).png" (a second upload of the same name) -> "tee-person.png"."""
    return re.sub(r" \(\d+\)(?=\.\w+$)", "", os.path.basename(name))

uploaded = files.upload()
for name, data in uploaded.items():
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            for member in z.infolist():
                if member.is_dir() or "__MACOSX" in member.filename or os.path.basename(member.filename).startswith("."):
                    continue
                open(os.path.join(SPIKE, plain(member.filename)), "wb").write(z.read(member))
    else:
        open(os.path.join(SPIKE, plain(name)), "wb").write(data)

# What came in, by NAME, with a check that each has its two needed files.
groups = {}
for f in sorted(os.listdir(SPIKE)):
    m = re.match(r"^(.*)-(person|garment|category|outline|mask|print|pose)\.\w+$", f)
    groups.setdefault(m.group(1) if m else "(not an input)", []).append(f)
ready = 0
for name, names in groups.items():
    if name == "(not an input)":
        print(f"   ignored: {', '.join(names)}")
        continue
    kinds = {re.match(r"^.*-(\w+)\.\w+$", f).group(1) for f in names}
    ok = {"person", "garment"} <= kinds
    ready += ok
    print(f"{'ok ' if ok else '-- '}{name}: {', '.join(names)}" + ("" if ok else "   <- needs NAME-person.png and NAME-garment.jpg"))
print(f"\n{ready} ready to run" if ready else "\nNothing to run yet: upload a NAME-person.png with its NAME-garment.jpg (or the spike.zip).")

## 4. Run

One model is loaded at a time (the T4 has 15 GB), every picture goes through it, then the next. Expect 30 to 90 seconds per picture per model after the first, which also loads the model.

In [ ]:
MODELS = "both"  #@param ["both", "fashn", "leffa"]
FASHN_PARSER = "off"  #@param ["off", "on"]
# off: FASHN runs without fashn-human-parser, whose licence (NVIDIA's SegFormer) allows research use only; shop photos are then read as flat-lay product shots.
STEPS = 30  #@param {type:"integer"}
SEED = 7  #@param {type:"integer"}

import importlib, os, subprocess, sys
os.environ["FASHN_PARSER"] = FASHN_PARSER
os.environ["HF_HOME"] = "/content/weights/hf"
sys.path[:0] = [p for p in ("/content/fitroom/script/modal", "/content/Leffa") if p not in sys.path]
# The latest engine from GitHub main, even on a re-run of a long session: pull, then reload what's loaded.
if os.path.isdir("/content/fitroom/.git"):
    pulled = subprocess.run(["git", "-C", "/content/fitroom", "pull", "-q", "--ff-only"], capture_output=True, text=True)
    print("engine:", subprocess.run(["git", "-C", "/content/fitroom", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip(),
          "" if pulled.returncode == 0 else f"(pull failed, running what's here: {pulled.stderr.strip()})")
for module in ("silhouette", "tryon_prep", "validate", "tryon_engine"):   # each before what imports it
    if module in sys.modules:
        importlib.reload(sys.modules[module])
from tryon_engine import run_sheet

models = ("fashn", "leffa") if MODELS == "both" else (MODELS,)
run_sheet(SPIKE, models=models, flat_lay=True, steps=STEPS, seed=SEED, weights="/content/weights", leffa_dir="/content/Leffa")

## 5. Look

The sheet: store photo | our render | FASHN | Leffa, with each gate's score and PASS or REFUSED under the AI columns. docs/ai-spike.md, "What to look for on the sheet", says how to read it.

In [ ]:
import json
from IPython.display import Image, display

display(Image(f"{SPIKE}/sheet.png"))
print(json.dumps(json.load(open(f"{SPIKE}/results.json")), indent=1))

## 6. Take it with you

Everything in the folder (your inputs, `NAME-fashn.png`, `NAME-leffa.png`, `sheet.png`, `results.json`) as one zip. The session and its files are gone once Colab closes it.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/fitroom-tryon-spike", "zip", SPIKE)
files.download("/content/fitroom-tryon-spike.zip")